In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Test that everything works
print("✅ All libraries imported successfully!")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

# Create a simple test plot
plt.figure(figsize=(8, 4))
plt.plot([0, 1, 2, 3], [0, 1, 4, 9])
plt.title("Test Plot - If you see this, matplotlib works!")
plt.xlabel("X axis")
plt.ylabel("Y axis")
plt.grid(True)
plt.show()

In [ ]:
# Setup paths for PTB-XL dataset
from pathlib import Path
import os

# Define project structure
PROJECT_DIR = Path("..")
DATA_DIR = PROJECT_DIR / "data" / "raw" / "ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3"

print("Project directory:", PROJECT_DIR)
print("Data directory:", DATA_DIR)
print("\nWaiting for dataset to be extracted to this location...")
print("After extraction, we'll load the data from here!")

In [ ]:
# Function to check if dataset is ready
def check_dataset_ready():
    """Check if PTB-XL dataset has been extracted"""
    if DATA_DIR.exists():
        csv_file = DATA_DIR / "ptbxl_database.csv"
        if csv_file.exists():
            print("✅ Dataset found and ready!")
            return True
        else:
            print("❌ Folder exists but CSV file not found")
            return False
    else:
        print("❌ Dataset folder not found yet")
        print(f"   Please extract ZIP to: {DATA_DIR.parent}")
        return False

# Test it (will show "not found" until you extract the ZIP)
check_dataset_ready()

In [ ]:
# ECG Preprocessing Functions
# We'll use these to clean and prepare ECG signals

import numpy as np
from scipy.signal import butter, filtfilt, resample

def resample_ecg(ecg, orig_fs, target_fs=250):
    """
    Resample ECG to target frequency
    ecg: numpy array (12, T) or (T, 12)
    orig_fs: original sampling frequency
    target_fs: target sampling frequency
    """
    # Make sure shape is (12, T)
    if ecg.shape[0] != 12 and ecg.shape[1] == 12:
        ecg = ecg.T
    
    T_orig = ecg.shape[1]
    T_new = int(T_orig * target_fs / orig_fs)
    ecg_resampled = resample(ecg, T_new, axis=1)
    return ecg_resampled

print("✅ resample_ecg function created!")

In [ ]:
def bandpass_filter(ecg, low=0.5, high=40, fs=250, order=4):
    """
    Apply bandpass filter to remove noise
    low: low frequency cutoff (removes baseline drift)
    high: high frequency cutoff (removes high-freq noise)
    """
    nyq = 0.5 * fs
    b, a = butter(order, [low / nyq, high / nyq], btype="band")
    filtered = filtfilt(b, a, ecg, axis=1)
    return filtered

print("✅ bandpass_filter function created!")

In [ ]:
def normalize_ecg(ecg, eps=1e-6):
    """
    Z-score normalization per lead
    Makes each lead have mean=0, std=1
    """
    mean = ecg.mean(axis=1, keepdims=True)
    std = ecg.std(axis=1, keepdims=True) + eps
    normalized = (ecg - mean) / std
    return normalized

print("✅ normalize_ecg function created!")

In [ ]:
def preprocess_raw_ecg(raw_ecg, orig_fs, target_fs=250):
    """
    Complete preprocessing pipeline:
    1. Resample to target frequency
    2. Apply bandpass filter
    3. Normalize
    """
    ecg = resample_ecg(raw_ecg, orig_fs, target_fs)
    ecg = bandpass_filter(ecg, fs=target_fs)
    ecg = normalize_ecg(ecg)
    return ecg

print("✅ Complete preprocessing pipeline ready!")
print("\nThis function will:")
print("  1. Resample ECG to 250 Hz")
print("  2. Remove noise with bandpass filter")
print("  3. Normalize each lead")

In [ ]:
%pip install wfdb

In [ ]:
import wfdb

In [ ]:
# Data Loading Functions
import pandas as pd
import wfdb  # Library for reading ECG waveform files

def load_ptbxl_metadata():
    """
    Load the PTB-XL metadata CSV file
    Contains patient info, diagnoses, and file paths
    """
    csv_path = DATA_DIR / "ptbxl_database.csv"
    df = pd.read_csv(csv_path)
    print(f"✅ Loaded metadata for {len(df)} ECG records")
    print(f"   Patients: {df['patient_id'].nunique()}")
    return df

print("✅ load_ptbxl_metadata function created!")

In [ ]:
def load_raw_ecg(df, idx, sampling_rate=100):
    """
    Load a single ECG waveform from the dataset
    
    df: metadata dataframe
    idx: index of the record to load
    sampling_rate: 100 or 500 Hz
    """
    if sampling_rate == 100:
        data = wfdb.rdsamp(str(DATA_DIR / df.loc[idx, 'filename_lr']))
    else:
        data = wfdb.rdsamp(str(DATA_DIR / df.loc[idx, 'filename_hr']))
    
    signal = data[0]  # ECG signal data
    return signal.T  # Return as (12, T) shape

print("✅ load_raw_ecg function created!")

In [ ]:
def create_mi_labels(df):
    """
    Create binary labels: 1 for MI (Myocardial Infarction), 0 for others
    This is our "critical" vs "non-critical" classification
    """
    # Extract diagnostic codes from the scp_codes column
    import ast
    
    def has_mi(scp_codes_str):
        """Check if record has MI diagnosis"""
        if pd.isna(scp_codes_str):
            return 0
        
        scp_codes = ast.literal_eval(scp_codes_str)
        
        # MI-related codes in PTB-XL
        mi_codes = ['IMI', 'AMI', 'LMI', 'PMI', 'ASMI', 'ILMI', 'ALMI', 'INJAS', 'INJAL', 'IPLMI', 'IPMI']
        
        for code in mi_codes:
            if code in scp_codes:
                return 1
        return 0
    
    df['is_mi'] = df['scp_codes'].apply(has_mi)
    
    print(f"✅ Labels created!")
    print(f"   MI cases (critical): {df['is_mi'].sum()}")
    print(f"   Non-MI cases: {(df['is_mi'] == 0).sum()}")
    
    return df

print("✅ create_mi_labels function created!")

In [ ]:
# Check if dataset is ready
check_dataset_ready()

In [ ]:
# Load the PTB-XL metadata
df = load_ptbxl_metadata()

# Show first few rows
df.head()

In [ ]:
# Create MI labels (critical vs non-critical)
df = create_mi_labels(df)

In [ ]:
# Let's look at a NORMAL ECG first
print("Loading a NORMAL ECG...")
normal_idx = df[df['is_mi'] == 0].index[0]
normal_ecg = load_raw_ecg(df, normal_idx, sampling_rate=100)

print(f"ECG shape: {normal_ecg.shape}")
print(f"This means: {normal_ecg.shape[0]} leads, {normal_ecg.shape[1]} samples")
print(f"Duration: {normal_ecg.shape[1] / 100} seconds at 100 Hz")

In [ ]:
# ECG Visualization Functions
import matplotlib.pyplot as plt

def plot_ecg(ecg, title="12-Lead ECG", sampling_rate=250):
    """
    Plot all 12 leads of an ECG
    ecg: numpy array of shape (12, T)
    """
    lead_names = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
    
    fig, axes = plt.subplots(12, 1, figsize=(15, 12))
    fig.suptitle(title, fontsize=16, fontweight='bold')
    
    time = np.arange(ecg.shape[1]) / sampling_rate
    
    for i in range(12):
        axes[i].plot(time, ecg[i], linewidth=0.8, color='black')
        axes[i].set_ylabel(lead_names[i], fontweight='bold')
        axes[i].grid(True, alpha=0.3)
        axes[i].set_xlim(0, time[-1])
        
        if i < 11:
            axes[i].set_xticks([])
    
    axes[11].set_xlabel('Time (seconds)', fontweight='bold')
    plt.tight_layout()
    plt.show()

print("✅ plot_ecg function created!")

In [ ]:
# Plot the normal ECG
plot_ecg(normal_ecg, title="Normal ECG (Non-Critical)", sampling_rate=100)

In [ ]:
# COMPLETE MI ECG Loading - Everything in one cell
from pathlib import Path
import pandas as pd
import ast
import wfdb

print("Loading a CRITICAL MI ECG (Heart Attack)...")

# Define paths
PROJECT_DIR = Path("..")
DATA_DIR = PROJECT_DIR / "data" / "raw" / "ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3"

# Load metadata
csv_path = DATA_DIR / "ptbxl_database.csv"
df = pd.read_csv(csv_path)
print(f"✅ Loaded {len(df)} ECG records")

# Create MI labels
def has_mi(scp_codes_str):
    if pd.isna(scp_codes_str):
        return 0
    scp_codes = ast.literal_eval(scp_codes_str)
    mi_codes = ['IMI', 'AMI', 'LMI', 'PMI', 'ASMI', 'ILMI', 'ALMI', 'INJAS', 'INJAL', 'IPLMI', 'IPMI']
    for code in mi_codes:
        if code in scp_codes:
            return 1
    return 0

df['is_mi'] = df['scp_codes'].apply(has_mi)
print(f"   MI cases: {df['is_mi'].sum()}")

# Load MI ECG
mi_idx = df[df['is_mi'] == 1].index[0]
data = wfdb.rdsamp(str(DATA_DIR / df.loc[mi_idx, 'filename_lr']))
mi_ecg = data[0].T  # Transpose to (12, T) shape

print(f"✅ ECG shape: {mi_ecg.shape}")
print(f"⚠️ Patient has Myocardial Infarction (Heart Attack)")

In [ ]:
# Plot the MI ECG
plot_ecg(mi_ecg, title="⚠️ CRITICAL: MI (Heart Attack) ECG ⚠️", sampling_rate=100)

In [ ]:
# COMPLETE Comparison: Load both Normal and MI ECGs
from pathlib import Path
import pandas as pd
import ast
import wfdb
import matplotlib.pyplot as plt
import numpy as np

# Define paths
PROJECT_DIR = Path("..")
DATA_DIR = PROJECT_DIR / "data" / "raw" / "ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3"

# Load metadata
csv_path = DATA_DIR / "ptbxl_database.csv"
df = pd.read_csv(csv_path)

# Create MI labels
def has_mi(scp_codes_str):
    if pd.isna(scp_codes_str):
        return 0
    scp_codes = ast.literal_eval(scp_codes_str)
    mi_codes = ['IMI', 'AMI', 'LMI', 'PMI', 'ASMI', 'ILMI', 'ALMI', 'INJAS', 'INJAL', 'IPLMI', 'IPMI']
    for code in mi_codes:
        if code in scp_codes:
            return 1
    return 0

df['is_mi'] = df['scp_codes'].apply(has_mi)

# Load NORMAL ECG
normal_idx = df[df['is_mi'] == 0].index[0]
data_normal = wfdb.rdsamp(str(DATA_DIR / df.loc[normal_idx, 'filename_lr']))
normal_ecg = data_normal[0].T  # Transpose to (12, T)

# Load MI ECG
mi_idx = df[df['is_mi'] == 1].index[0]
data_mi = wfdb.rdsamp(str(DATA_DIR / df.loc[mi_idx, 'filename_lr']))
mi_ecg = data_mi[0].T  # Transpose to (12, T) - FIXED!

# Plot comparison
fig, axes = plt.subplots(2, 1, figsize=(15, 6))

# Normal ECG - Lead II
axes[0].plot(normal_ecg[1], linewidth=0.8, color='green')
axes[0].set_title('✅ NORMAL ECG - Lead II', fontweight='bold', fontsize=14, color='green')
axes[0].grid(True, alpha=0.3)
axes[0].set_ylabel('Amplitude', fontweight='bold')

# MI ECG - Lead II
axes[1].plot(mi_ecg[1], linewidth=0.8, color='red')
axes[1].set_title('⚠️ CRITICAL MI ECG - Lead II', fontweight='bold', fontsize=14, color='red')
axes[1].grid(True, alpha=0.3)
axes[1].set_ylabel('Amplitude', fontweight='bold')
axes[1].set_xlabel('Sample', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n🔍 Key differences our AI will learn to detect:")
print("   ✅ Normal: Regular rhythm, normal ST segment")
print("   ⚠️  MI: ST elevation/depression, T-wave changes, abnormal QRS")
print("\n💡 This is what saves lives in the ED!")

In [ ]:
# Simple PyTorch Dataset for ECG Training
from torch.utils.data import Dataset
import torch
import numpy as np

class PTBXLDataset(Dataset):
    """
    Simple PyTorch Dataset for PTB-XL ECG data
    """
    def __init__(self, ecg_list, labels):
        """
        ecg_list: list of numpy arrays (already loaded ECGs)
        labels: list of labels (0 or 1)
        """
        self.ecgs = ecg_list
        self.labels = labels
        
    def __len__(self):
        return len(self.ecgs)
    
    def __getitem__(self, idx):
        ecg = self.ecgs[idx]
        label = self.labels[idx]
        
        # Convert to PyTorch tensors
        ecg_tensor = torch.FloatTensor(ecg)
        label_tensor = torch.LongTensor([label])[0]
        
        return ecg_tensor, label_tensor

print("✅ Simple PTBXLDataset class created!")

In [ ]:
# ========================================
# TODAY'S ACCOMPLISHMENTS - WEEK 1 COMPLETE! 🎉
# ========================================

print("=" * 60)
print("✅ WEEK 1 COMPLETE - FOUNDATION BUILT!")
print("=" * 60)

accomplishments = """
✅ Environment Setup:
   • Virtual environment created
   • All libraries installed (numpy, pandas, torch, etc.)
   • Jupyter Notebook working

✅ Data Acquisition:
   • PTB-XL dataset downloaded (21,799 ECG records)
   • 5,461 MI cases identified
   • 16,338 Normal cases identified
   • Data extracted and verified

✅ Data Exploration:
   • Loaded and visualized real ECG signals
   • Compared Normal vs MI ECGs visually
   • Understood key differences (ST changes, T-waves)

✅ Code Components Ready:
   • Preprocessing functions (resample, filter, normalize)
   • Data loading functions
   • Label creation for MI detection
   • Visualization functions
   • PyTorch Dataset class (needs DLL fix)

📊 NEXT STEPS - WEEK 2:
   1. Fix PyTorch DLL issue (install Visual C++)
   2. Build 1D-CNN model architecture
   3. Train the model on 1000 samples
   4. Evaluate accuracy & metrics
   5. Save the trained model

💡 COMMERCIAL POTENTIAL:
   • Core AI for ED triage ✅
   • 90%+ accuracy target
   • Foundation for $100K+ product
"""

print(accomplishments)
print("=" * 60)
print("🚀 YOU'RE ON TRACK TO BUILD SOMETHING VALUABLE!")
print("=" * 60)

In [ ]:
# Load ECG data into memory (avoiding DLL issue)
from pathlib import Path
import pandas as pd
import ast
import wfdb
import numpy as np
from tqdm import tqdm

print("Loading ECG data into memory...")

# Define paths
PROJECT_DIR = Path("..")
DATA_DIR = PROJECT_DIR / "data" / "raw" / "ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3"

# Load metadata
csv_path = DATA_DIR / "ptbxl_database.csv"
df = pd.read_csv(csv_path)

# Create MI labels
def has_mi(scp_codes_str):
    if pd.isna(scp_codes_str):
        return 0
    scp_codes = ast.literal_eval(scp_codes_str)
    mi_codes = ['IMI', 'AMI', 'LMI', 'PMI', 'ASMI', 'ILMI', 'ALMI', 'INJAS', 'INJAL', 'IPLMI', 'IPMI']
    for code in mi_codes:
        if code in scp_codes:
            return 1
    return 0

df['is_mi'] = df['scp_codes'].apply(has_mi)

# Use subset: 400 MI, 400 Normal (800 total for faster training)
mi_samples = df[df['is_mi'] == 1].sample(n=400, random_state=42)
normal_samples = df[df['is_mi'] == 0].sample(n=400, random_state=42)
df_subset = pd.concat([mi_samples, normal_samples]).sample(frac=1, random_state=42).reset_index(drop=True)

# Load all ECGs into memory
ecg_data = []
labels = []

print(f"Loading {len(df_subset)} ECG records...")

for idx in tqdm(range(len(df_subset))):
    try:
        filename = df_subset.loc[idx, 'filename_lr']
        data = wfdb.rdsamp(str(DATA_DIR / filename))
        ecg = data[0].T  # Shape: (12, 1000)
        label = df_subset.loc[idx, 'is_mi']
        
        ecg_data.append(ecg)
        labels.append(label)
    except Exception as e:
        print(f"Error loading index {idx}: {e}")
        continue

ecg_data = np.array(ecg_data)
labels = np.array(labels)

print(f"\n✅ Loaded {len(ecg_data)} ECGs successfully!")
print(f"   ECG shape: {ecg_data.shape}")
print(f"   MI cases: {labels.sum()}")
print(f"   Normal cases: {(labels == 0).sum()}")

In [ ]:
# Split data: 70% train, 15% validation, 15% test
from sklearn.model_selection import train_test_split

# First split: 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    ecg_data, labels, test_size=0.3, random_state=42, stratify=labels
)

# Second split: 15% validation, 15% test (from the 30% temp)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print("✅ Data split complete!")
print(f"   Training:   {len(X_train)} samples ({y_train.sum()} MI, {(y_train==0).sum()} Normal)")
print(f"   Validation: {len(X_val)} samples ({y_val.sum()} MI, {(y_val==0).sum()} Normal)")
print(f"   Test:       {len(X_test)} samples ({y_test.sum()} MI, {(y_test==0).sum()} Normal)")

In [ ]:
# Create PyTorch datasets and dataloaders
import torch
from torch.utils.data import TensorDataset, DataLoader

# Convert to PyTorch tensors
X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.LongTensor(y_train)

X_val_t = torch.FloatTensor(X_val)
y_val_t = torch.LongTensor(y_val)

X_test_t = torch.FloatTensor(X_test)
y_test_t = torch.LongTensor(y_test)

# Create datasets
train_dataset = TensorDataset(X_train_t, y_train_t)
val_dataset = TensorDataset(X_val_t, y_val_t)
test_dataset = TensorDataset(X_test_t, y_test_t)

# Create dataloaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("✅ DataLoaders created!")
print(f"   Batch size: {batch_size}")
print(f"   Training batches: {len(train_loader)}")
print(f"   Validation batches: {len(val_loader)}")
print(f"   Test batches: {len(test_loader)}")

In [ ]:
# 1D-CNN Model for ECG Classification
import torch
import torch.nn as nn

class ECG_CNN(nn.Module):
    def __init__(self):
        super(ECG_CNN, self).__init__()
        
        # Convolutional layers
        self.conv1 = nn.Conv1d(in_channels=12, out_channels=32, kernel_size=7, padding=3)
        self.bn1 = nn.BatchNorm1d(32)
        self.pool1 = nn.MaxPool1d(2)
        
        self.conv2 = nn.Conv1d(32, 64, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(64)
        self.pool2 = nn.MaxPool1d(2)
        
        self.conv3 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(128)
        self.pool3 = nn.MaxPool1d(2)
        
        # Global average pooling
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        
        # Fully connected layers
        self.fc1 = nn.Linear(128, 64)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(64, 2)  # 2 classes: Normal (0) or MI (1)
        
    def forward(self, x):
        # x shape: (batch, 12, 1000)
        
        x = self.pool1(torch.relu(self.bn1(self.conv1(x))))
        x = self.pool2(torch.relu(self.bn2(self.conv2(x))))
        x = self.pool3(torch.relu(self.bn3(self.conv3(x))))
        
        x = self.global_pool(x)  # (batch, 128, 1)
        x = x.squeeze(-1)        # (batch, 128)
        
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        
        return x

# Create model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ECG_CNN().to(device)

print("✅ ECG-CNN Model created!")
print(f"   Device: {device}")
print(f"\nModel architecture:")
print(model)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"\n   Total parameters: {total_params:,}")

In [ ]:
# Training setup
import torch.optim as optim

# Loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)

print("✅ Training setup complete!")
print(f"   Loss function: CrossEntropyLoss")
print(f"   Optimizer: Adam (lr=0.001)")
print(f"   Scheduler: ReduceLROnPlateau")

In [ ]:
# Training function
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for ecgs, labels in loader:
        ecgs = ecgs.to(device)
        labels = labels.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(ecgs)
        loss = criterion(outputs, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Statistics
        running_loss += loss.item() * ecgs.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    epoch_loss = running_loss / total
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc

# Validation function
def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for ecgs, labels in loader:
            ecgs = ecgs.to(device)
            labels = labels.to(device)
            
            outputs = model(ecgs)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * ecgs.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    epoch_loss = running_loss / total
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc

print("✅ Training functions ready!")

In [ ]:
# Train the model!
import time

num_epochs = 20
best_val_acc = 0

print("🚀 STARTING TRAINING!")
print("=" * 60)

for epoch in range(1, num_epochs + 1):
    start_time = time.time()
    
    # Train
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    
    # Validate
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    
    # Learning rate scheduling
    scheduler.step(val_loss)
    
    epoch_time = time.time() - start_time
    
    # Print progress
    print(f"Epoch {epoch:2d}/{num_epochs} | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}% | "
          f"Time: {epoch_time:.1f}s")
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_ecg_model.pt')
        print(f"   ✅ New best model saved! (Val Acc: {val_acc:.2f}%)")

print("=" * 60)
print(f"🎉 TRAINING COMPLETE!")
print(f"   Best Validation Accuracy: {best_val_acc:.2f}%")